# CS131 — Stage 2 Final Run (Day 4)

**Config:** ResNet-34 encoder · lr=2e-5 · γ=2 · inv-sqrt-freq weights · batch 8 · 20 epochs · stratified sampler  


In [ ]:
# ── paths on Drive ──────────────────────────────────────────────────────────
REPO_URL        = 'https://github.com/Tantan4321/cs131_final_project.git'
PROJECT_DIR     = '/content/final_project'
DRIVE_BUNDLE    = '/content/drive/MyDrive/xbd_bundle'   # xbd_images.tar.gz, xbd_cache.tar.gz, loc.pt
DRIVE_CKPT_DIR  = '/content/drive/MyDrive/xbd_ckpts'    # checkpoints written here (persist across disconnects)
DRIVE_OUT_DIR   = '/content/drive/MyDrive/xbd_outputs'  # figs + logs mirrored here at the end

# ── training hyperparameters ────────────────────────────────────────────────
EPOCHS      = 20
BATCH_SIZE  = 8
LR          = '2e-5'
GAMMA       = 2.0
WEIGHTS     = 'inv-sqrt-freq'
ENCODER     = 'resnet34'
SEED        = 131
RUN_NAME    = 'final'

!nvidia-smi -L

## 1. Clone repo (no LFS) + install deps

In [ ]:
import os
if not os.path.exists(PROJECT_DIR):
    # GIT_LFS_SKIP_SMUDGE=1 clones without materialising any LFS objects.
    # All data comes from Drive instead.
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 {REPO_URL} {PROJECT_DIR}
else:
    print(f'{PROJECT_DIR} already exists, skipping clone')

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
print('deps installed')

## 2. Mount Drive + unpack training data

Unpacks `xbd_images.tar.gz` and `xbd_cache.tar.gz` from Drive into `train/` inside the project.  
Each archive is skipped if the target directory already has the expected content (safe to re-run).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib, tarfile, time

bundle = pathlib.Path(DRIVE_BUNDLE)
project = pathlib.Path(PROJECT_DIR)

def unpack_if_needed(archive_name, sentinel_dir):
    """Extract archive into project root unless sentinel_dir already exists."""
    sentinel = project / sentinel_dir
    if sentinel.exists() and any(sentinel.iterdir()):
        count = sum(1 for _ in sentinel.rglob('*'))
        print(f'[skip] {archive_name}: {sentinel} already has {count} files')
        return
    archive = bundle / archive_name
    assert archive.exists(), f'{archive} not found -- upload it to Drive first'
    print(f'Unpacking {archive_name} ({archive.stat().st_size/1e9:.1f} GB) ...')
    t0 = time.time()
    with tarfile.open(archive, 'r:gz') as tf:
        tf.extractall(project)
    print(f'  done in {(time.time()-t0)/60:.1f} min')

unpack_if_needed('xbd_images.tar.gz', 'train/xBD')
unpack_if_needed('xbd_cache.tar.gz',  'train/cache')

# Quick sanity check
n_pre  = len(list((project / 'train/xBD').rglob('*_pre_disaster.png')))
n_npy  = len(list((project / 'train/cache/dmg').rglob('*.npy')))
print(f'train/xBD: {n_pre} pre-disaster images')
print(f'train/cache/dmg: {n_npy} damage mask .npy files')

## 3. Set up checkpoints → Drive symlink

Copies `loc.pt` from the Drive bundle into `outputs/checkpoints/`, then symlinks that directory to Drive so every epoch's checkpoint persists automatically.

In [ ]:
import shutil, pathlib

drive_ckpts = pathlib.Path(DRIVE_CKPT_DIR)
drive_ckpts.mkdir(parents=True, exist_ok=True)

# Copy loc.pt from bundle to Drive (once)
loc_dst = drive_ckpts / 'loc.pt'
if not loc_dst.exists():
    loc_src = pathlib.Path(DRIVE_BUNDLE) / 'loc.pt'
    assert loc_src.exists(), f'{loc_src} not found -- upload loc.pt to Drive bundle'
    shutil.copy2(loc_src, loc_dst)
    print(f'Copied loc.pt -> Drive ({loc_dst.stat().st_size/1e6:.1f} MB)')
else:
    print(f'loc.pt already in Drive ({loc_dst.stat().st_size/1e6:.1f} MB)')

# Symlink outputs/checkpoints -> Drive
local_ckpts = pathlib.Path(PROJECT_DIR) / 'outputs' / 'checkpoints'
pathlib.Path(PROJECT_DIR, 'outputs').mkdir(exist_ok=True)
if local_ckpts.exists() and not local_ckpts.is_symlink():
    # Move any existing checkpoints to Drive then replace with symlink
    for f in local_ckpts.glob('*.pt'):
        dst = drive_ckpts / f.name
        if not dst.exists():
            shutil.copy2(f, dst)
    shutil.rmtree(local_ckpts)

if not local_ckpts.exists():
    local_ckpts.symlink_to(drive_ckpts)

print(f'outputs/checkpoints -> {drive_ckpts}')
!ls -lh {DRIVE_CKPT_DIR}/*.pt 2>/dev/null || echo '(no .pt files yet)'

## 4. Precompute stratified sampler weights

Reads every damage mask once to build `outputs/stratified_weights.pt`. Cached — safe to skip on resume.

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from src.data.xbd_dataset import list_pairs, split_pairs, compute_tile_weights
from src.paths import OUTPUTS

all_pairs = list_pairs()
train_pairs, val_pairs = split_pairs(all_pairs)
print(f'train={len(train_pairs)}  val={len(val_pairs)}')

cache = str(OUTPUTS / 'stratified_weights.pt')
weights = compute_tile_weights(train_pairs, cache_file=cache)
print(f'Tile weights shape: {weights.shape}  min={weights.min():.4f}  max={weights.max():.4f}')

## 5. Train — 20 epochs

Saves `dmg_final.pt` (best val macro-F1) and `dmg_final_last.pt` (resume state) to Drive every epoch.  
> **After a disconnect:** re-run cells 1–3, then re-run this cell. The `--resume` flag picks up from the last completed epoch.

In [ ]:
!python -m src.train_dmg \
    --epochs      {EPOCHS}     \
    --batch-size  {BATCH_SIZE} \
    --lr          {LR}         \
    --gamma       {GAMMA}      \
    --weights     {WEIGHTS}    \
    --encoder     {ENCODER}    \
    --seed        {SEED}       \
    --run-name    {RUN_NAME}   \
    --stratified

In [ ]:
# After a runtime disconnect: re-run cells 1-3, then run THIS cell instead of the one above.
# --epochs is the absolute target (20 = finish at epoch 20, not +20 more).
!python -m src.train_dmg \
    --epochs      {EPOCHS}     \
    --batch-size  {BATCH_SIZE} \
    --lr          {LR}         \
    --gamma       {GAMMA}      \
    --weights     {WEIGHTS}    \
    --encoder     {ENCODER}    \
    --seed        {SEED}       \
    --run-name    {RUN_NAME}   \
    --stratified              \
    --resume

## 6. Training curves

Three panels: loss (train + val) · train per-class F1 · val per-class F1.

In [ ]:
from IPython.display import Image, display
display(Image(f'{PROJECT_DIR}/outputs/figs/dmg_{RUN_NAME}_training_curves.png'))

## 7. Damage prediction gallery

8-row grid on held-out santa-rosa-wildfire tiles:  
`pre | post | ground truth | prediction | prediction overlaid on post`

In [ ]:
!python -m src.tools.visualize_damage \
    --ckpt {PROJECT_DIR}/outputs/checkpoints/dmg_{RUN_NAME}.pt \
    --n 8

display(Image(f'{PROJECT_DIR}/outputs/figs/dmg_predictions.png'))

## 8. Confusion matrix

Row-normalised 4×4 confusion over all val pixels.

In [ ]:
!python -m src.tools.eval_dmg \
    --ckpt {PROJECT_DIR}/outputs/checkpoints/dmg_{RUN_NAME}.pt

display(Image(f'{PROJECT_DIR}/outputs/figs/dmg_confusion.png'))

## 9. Persist outputs to Drive

In [ ]:
import pathlib, shutil

out_drive = pathlib.Path(DRIVE_OUT_DIR)
(out_drive / 'figs').mkdir(parents=True, exist_ok=True)
(out_drive / 'logs').mkdir(parents=True, exist_ok=True)

figs_src = pathlib.Path(PROJECT_DIR) / 'outputs' / 'figs'
logs_src = pathlib.Path(PROJECT_DIR) / 'outputs' / 'logs'

for f in figs_src.glob('*.png'):
    shutil.copy2(f, out_drive / 'figs' / f.name)
for f in logs_src.glob('*.json'):
    shutil.copy2(f, out_drive / 'logs' / f.name)

print('Outputs mirrored to Drive:')
!ls {DRIVE_OUT_DIR}/figs
!ls {DRIVE_OUT_DIR}/logs